In [ ]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_mistralai import ChatMistralAI

load_dotenv()

In [ ]:
#loaidng llm into memory...
llm = ChatMistralAI(
    model_name = "mistral-medium-3-5",
    temperature= 0.3
)

#loading model into memory...
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
#PDF data load into cell...
data = PyPDFLoader("test.pdf")
docs = data.load()

#splitting them into chunks...
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs)
if chunks:
    vectorstore = Chroma.from_documents(
        documents = chunks,
        embedding = embedding_model,
        persist_directory = "../chroma-DB"
    )
else:
    print("chunks not created...")

In [ ]:
similar_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k":5,
        "fetch_k":10,
        "lambda_mult":0.5
    }
)

In [ ]:
retriever_response = similar_retriever.invoke("explain the Trace Operator?")
if retriever_response:
    for doc in retriever_response:
        print(doc.page_content)
else:
    print('response not generated...')